# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get the available record sets from metadata
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs['@id']}")

# For each record set, print the fields and columns
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs['@id']}) Fields:")
    for field in rs.fields:
        print(f"  - Field name: {field.name}, @id: {field['@id']}, dataType: {field.data_type}")
    print("  Columns:")
    if hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            print(f"    * Column name: {col.name}, @id: {col['@id']}, source: {col.source}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Get the list of record set @ids
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns for the first record set
first_record_set_id = record_sets_ids[0]
print(f"Columns in record set {first_record_set_id}: {dataframes[first_record_set_id].columns.tolist()}")
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field for analysis from the first record set
# Using @id for the GAD-7 score field as an example (update as needed based on actual IDs)
numeric_field_id = None
for field in dataset.metadata.record_sets[0].fields:
    if 'gad7_score' in field.name.lower() or field.data_type == 'schema:Float' or field.data_type == 'schema:Integer':
        numeric_field_id = field['@id']
        break

if numeric_field_id is not None:
    df = dataframes[first_record_set_id]
    # If the actual column name isn't the @id, map the @id to column name
    actual_numeric_col = None
    for col in df.columns:
        if numeric_field_id in col:
            actual_numeric_col = col
            break
        if 'gad7' in col.lower() or 'score' in col.lower():
            actual_numeric_col = col
            break

    # Filter example: records with scores greater than threshold
    threshold = 10
    filtered_df = df[df[actual_numeric_col] > threshold]
    print(f"Filtered records with {actual_numeric_col} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{actual_numeric_col}_normalized"] = (filtered_df[actual_numeric_col] - filtered_df[actual_numeric_col].mean()) / filtered_df[actual_numeric_col].std()
    print(f"Normalized {actual_numeric_col} for filtered records:")
    print(filtered_df[[actual_numeric_col, f"{actual_numeric_col}_normalized"]].head())

    # Group by another categorical field, e.g., gender
    group_field_id = None
    for field in dataset.metadata.record_sets[0].fields:
        if 'gender' in field.name.lower():
            group_field_id = field['@id']
            break
    actual_group_col = None
    if group_field_id:
        for col in df.columns:
            if group_field_id in col:
                actual_group_col = col
                break
            if 'gender' in col.lower():
                actual_group_col = col
                break

    if actual_group_col and actual_group_col in filtered_df.columns:
        grouped_df = filtered_df.groupby(actual_group_col)[actual_numeric_col].mean().reset_index()
        print(f"Grouped data by {actual_group_col} (mean {actual_numeric_col}):")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field (e.g., GAD-7 score)
if numeric_field_id is not None and actual_numeric_col is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[actual_numeric_col].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {actual_numeric_col}")
    plt.xlabel(actual_numeric_col)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by gender if available
    if actual_group_col:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=actual_group_col, y=actual_numeric_col, data=df)
        plt.title(f"{actual_numeric_col} by {actual_group_col}")
        plt.xlabel(actual_group_col)
        plt.ylabel(actual_numeric_col)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded metadata and records from a Croissant AI-ready survey dataset covering mental health indicators in Kilifi County, Kenya.
- Explored available record sets and their associated fields using the unique `@id` identifiers for reference and extraction.
- Performed filtering and normalization on key numeric fields (e.g., GAD-7 scores) and grouped by demographic fields (e.g., gender) using `@id`-driven referencing.
- Visualized score distributions and demographic relationships, supporting further downstream analysis and community insight into mental health trends.